In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time 

#COLLECTION VUOTA PER RACCOGLIERE LE INFORMAZIONI
world_movies = []

#specifico l'header in una variabile
headers = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

#cambio dinamicamente la variabile anno dell'url e ottenere i dati del decennio interessato
#inizio dal 2015 aggiungendo di 1 fino al 2025 
for anno in range(2015, 2026, +1):

    #CODICE ACCESSO HTML SITO WEB BOXOFFICEMOJO.COM
    #scompongo base_url e list_details così da poter cambiare l'anno di selezione
    base_url = "https://www.boxofficemojo.com" 
    url_details = f"/year/world/{anno}/" 
    full_url = base_url + url_details 

    #ricreo una sessione aperta per ogni anno 
    session = requests.Session()
    session.headers.update(headers)

    #RICHIESTA AL SITO WEB 
    #scarico pagina web
    request = session.get(full_url, headers=headers) 
    #passo a beautifulsoup
    soup = BeautifulSoup(request.content, "html.parser")

    #TROVA OGNI RIGA DELLA PAGINA
    world_containers = soup.find_all("tr")

    #LOOP
    for i, wm in enumerate(world_containers, start=0) :
        #titoli
        wtitle = wm.find("td", class_ = "a-text-left mojo-field-type-release_group")

        #worldwide, domestic, foreign grosses
        wgross_cells = wm.find_all("td", class_ = "a-text-right mojo-field-type-money")

        #domestic, foreign percents
        percent_cells = wm.find_all("td", class_ = "a-text-right mojo-field-type-percent")

        #dato che gli elementi che mi servono hanno la stessa classe, li recupero tutti 
        #e poi li divido in base alle posizioni
        if len(wgross_cells)>=3:
            worldwide = wgross_cells[0]
            domestic = wgross_cells[1]
            foreign = wgross_cells[2]

        if len(percent_cells)>=2:
            d_percent = percent_cells[0]   
            f_percent = percent_cells[1] 

        #mi accerto che non ci siano vuoti e li aggiungo alla lista
        if wtitle and worldwide and domestic and foreign and d_percent and f_percent:
            world_movies.append({ "n_rank" : i,
                              "year_rank" : anno,
                              "title" : wtitle.text.strip().upper(), 
                              "worldwide_gross" : worldwide.text.strip(),
                              "domestic_gross" : domestic.text.strip(),
                              "domestic_perc" : d_percent.text.strip(),
                              "foreign_gross" : foreign.text.strip(),
                              "foreign_perc" : f_percent.text.strip()
                            })
                
worldwide = pd.DataFrame(world_movies)

#PULIZIA DATI
#preferisco pulire i dati, in quanto negli incassi è presente la valuta $ e le virgole tipiche del formato americano, e nelle percentuali vi è %

#seleziono le colonne interessate alla pulizia
gross_cols = ["worldwide_gross", "domestic_gross", "foreign_gross"]
perc_cols = ["domestic_perc", "foreign_perc"]

#per ogni colonna interessata, sostiuisco $, virgola e %
for col in gross_cols:
    worldwide[col] = worldwide[col].str.replace("$", "", regex=False).str.replace(",", "", regex=False)

for col in perc_cols:
    worldwide[col] = worldwide[col].str.replace("%", "", regex=False).str.replace("<0.1", "0", regex=False)

#sostituisco in generale eventuali valori non specificati 
worldwide = worldwide.replace("-", "0")

worldwide.to_csv("fact_intgross.csv", encoding="utf-8", index=False,)

worldwide

,n_rank,year_rank,title,worldwide_gross,domestic_gross,domestic_perc,foreign_gross,foreign_perc
0,1,2015,STAR WARS: EPISODE VII - THE FORCE AWAKENS,2068223624,936662225,45.3,1131561399,54.7
1,2,2015,JURASSIC WORLD,1670400637,652270625,39,1018130012,61
2,3,2015,FURIOUS 7,1515047671,353007020,23.3,1162040651,76.7
3,4,2015,AVENGERS: AGE OF ULTRON,1402805868,459005868,32.7,943800000,67.3
4,5,2015,MINIONS,1159398397,336045770,29,823352627,71
...,...,...,...,...,...,...,...,...
2195,196,2025,GRACE,10015229,132311,1.3,9882918,98.7
2196,197,2025,SISU: ROAD TO REVENGE,9811703,4544481,46.3,5267222,53.7
2197,198,2025,CHASSE GARDÉE 2,9788446,0,0,9788446,100
2198,199,2025,DETECTIVE KIEN: THE HEADLESS HORROR,9768694,0,0,9768694,100


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time 

italy_movies = []

headers = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

for anno in range(2015, 2026, +1):

    base_url = "https://www.boxofficemojo.com" 
    url_details2 = f"/weekend/by-year/{anno}/?area=IT/" 
    full_url2 = base_url + url_details2 

    session = requests.Session()
    session.headers.update(headers)
    
    request2 = session.get(full_url2, headers=headers) 
    soup2 = BeautifulSoup(request2.content, "html.parser")

    italy_containers = soup2.find_all("tr")

    for i, im in enumerate(italy_containers, start=0) :

        ititle = im.find("td", class_ = "a-text-left mojo-field-type-release mojo-cell-wide")
        
        itagross_cells = im.find("td", class_ = "a-text-right mojo-field-type-money mojo-estimatable")
        
        ita_dates = im.find("td", class_ = "a-text-left mojo-header-column mojo-truncate mojo-field-type-date_interval mojo-sort-column")

        if ititle and itagross_cells and ita_dates:
            ita_dates_num = ita_dates.text.strip()

            #prendo in considerazione solo i weekend normali, senza festività incluse nella data
            #le date con festività hanno una lunghezza maggiore a 11 
            if len(ita_dates_num) <=11: 
                italy_movies.append({  "year_rank" : anno,
                                       "dates" : ita_dates.text.strip(),
                                       "title" : ititle.text.strip().upper(), 
                                       "italy_gross" : itagross_cells.text.strip()
                                    })

italy = pd.DataFrame(italy_movies)

#PULIZIA DATI  
itagross_cols = ["italy_gross"]

for col in itagross_cols:
    italy[col] = italy[col].str.replace("$", "", regex=False).str.replace(",", "", regex=False)

italy = italy.replace("-", "0")

italy.to_csv("fact_itagross.csv", encoding="utf-8", index=False)

italy

,year_rank,dates,title,italy_gross
0,2015,Dec 25-27,STAR WARS: EPISODE VII - THE FORCE AWAKENS,296447415
1,2015,Dec 18-20,STAR WARS: EPISODE VII - THE FORCE AWAKENS,313284961
2,2015,Dec 11-13,THE HUNGER GAMES: MOCKINGJAY - PART 2,77446378
3,2015,Nov 20-22,THE HUNGER GAMES: MOCKINGJAY - PART 2,173610096
4,2015,Nov 13-15,SPECTRE,108047802
...,...,...,...,...
439,2025,Feb 7-9,DOG MAN,54280219
440,2025,Jan 24-26,FLIGHT RISK,64538641
441,2025,Jan 17-19,MUFASA: THE LION KING,77670440
442,2025,Jan 10-12,DEN OF THIEVES: PANTERA,78558448


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time 

italinks = []
itaunique_linklist = []
itamovie_dati = []

headers = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

for anno in range(2015, 2026, +1):

    base_url = "https://www.boxofficemojo.com" 
    url_details2 = f"/weekend/by-year/{anno}/?area=IT/" 
    full_url2 = base_url + url_details2

    session = requests.Session()
    session.headers.update(headers)

    request2 = session.get(full_url2, headers=headers) 
    soup2 = BeautifulSoup(request2.content, "html.parser")

    #PRIMO SCRAPING
    ilink_containers = soup2.find_all("tr")

    #PER OGNI TITOLO DI FILM, TROVO IL LINK DELLA SCHEDA FILM
    for il in ilink_containers:

        #aggiungo il "filtro" per i weekend senza festività
        ita_dates = il.find("td", class_ = "a-text-left mojo-header-column mojo-truncate mojo-field-type-date_interval mojo-sort-column")
        
        if ita_dates: 
            ita_dates_num = ita_dates.text.strip()
            
            if len(ita_dates_num) <=11: 

                title_cell = il.find("td", class_ = "a-text-left mojo-field-type-release mojo-cell-wide")

                if title_cell:
                    ita_link_tag = title_cell.find("a", class_ = "a-link-normal")
            
                    if ita_link_tag:
                        itamovie_link = ita_link_tag.get("href")
                        italast_link = itamovie_link.split("?")[0]

                        if italast_link not in itaunique_linklist:
                            italinks.append({ "Link" : italast_link
                                        })
                        
                        #creo un ciclo per salvare in una nuova lista i link unici senza duplicati 
                        #dato che nella pagina originale, i film spesso si ripetono nei weekend
                            for link in italinks:
                                if link not in itaunique_linklist:
                                    itaunique_linklist.append(link)


#SECONDO SCRAPING 

#PER OGNI LINK, TROVO LE ALTRE INFORMAZIONI
#per ogni link unico nella lista del primo scraping, creo il nuovo url
for unique in itaunique_linklist:

    base_url = "https://www.boxofficemojo.com"
    full_url3 = base_url + unique["Link"]
    
    session = requests.Session()
    session.headers.update(headers)

    request3 = session.get(full_url3, headers=headers, timeout=30)
    soup3 = BeautifulSoup(request3.content, "html.parser")

    #mi prendo il titolo principale del film
    itamovie_title = soup3.find("h1", class_= "a-size-extra-large")

    #accedo alla pagina selezionando tutti i div da cui prendere la tabella 
    itamojo_containers = soup3.find("div", class_="a-section a-spacing-none mojo-gutter mojo-summary-table")

    if itamojo_containers and itamovie_title:

        itamojo_table = itamojo_containers.find("div", class_ = "a-section a-spacing-none mojo-summary-values mojo-hidden-from-mobile")

        if itamojo_table:

                #tramite le etichette specifiche dello span trovo il testo che mi serve 
                itatag_distributor = itamojo_table.find("span", string= "Distributor")

                if itatag_distributor:
                    itadistributor_container = itatag_distributor.find_next("span")
                    #prende solo il primo span 
                    itamovie_distributor = itadistributor_container.contents[0].strip()
                else:
                    itamovie_distributor = "Not classified"    

                itatag_genre = itamojo_table.find("span", string="Genres")
                
                if itatag_genre:
                    itageneral_genre = itatag_genre.find_next("span").get_text()
                    itaclean_genre = itageneral_genre.split()
                    itamovie_genre = ", ".join(itaclean_genre)
                else:
                    itamovie_genre = "Not classified"

            
                itamovie_dati.append({ "title": itamovie_title.text.strip().upper(),
                                       "distributor" : itamovie_distributor,
                                       "genre" : itamovie_genre
                                })
                time.sleep(1)

itamovies = pd.DataFrame(itamovie_dati)

#salvo tutto in un csv momentaneo
itamovies.to_csv("dati_grezzi_scraping/anagrafica_ita.csv", encoding="utf-8", index=False)

itamovies


,title,distributor,genre
0,STAR WARS: EPISODE VII - THE FORCE AWAKENS,Walt Disney Studios Motion Pictures,"Action, Adventure, Sci-Fi"
1,THE HUNGER GAMES: MOCKINGJAY - PART 2,Lionsgate,"Action, Adventure, Sci-Fi, Thriller"
2,SPECTRE,Sony Pictures Releasing,"Action, Adventure, Thriller"
3,THE MARTIAN,Twentieth Century Fox,"Adventure, Drama, Sci-Fi"
4,GOOSEBUMPS,Sony Pictures Releasing,"Adventure, Comedy, Family, Fantasy, Horror, Th..."
...,...,...,...
283,CAPTAIN AMERICA: BRAVE NEW WORLD,Walt Disney Studios Motion Pictures,"Action, Adventure, Sci-Fi"
284,DOG MAN,Universal Pictures,"Action, Adventure, Animation, Comedy, Crime, F..."
285,FLIGHT RISK,Lionsgate,"Action, Crime, Drama, Thriller"
286,MUFASA: THE LION KING,Walt Disney Studios Motion Pictures,"Adventure, Animation, Drama, Family, Musical"


In [14]:
#GENERI FILM CLASSIFICA INTERNAZIONALE
#per recuperare il genere dalla classifica internazionale, ho bisogno di fare uno scraping di terzo livello
#dato che quando dalla classifica si clicca il titolo di un film, non porta direttamente alla scheda film come per l'italia,
#ma porta alla schermata in cui si vedono gli incassi
#il genere è in un'altra sezione della schermata incassi, per cui si deve fare un terzo scraping

import requests
import pandas as pd
from bs4 import BeautifulSoup
import time 

worldlinks = []
lastworldlinks = []
worldmovie_dati = []

headers = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}


#PRIMO SCRAPING
anno = 2025
base_url = "https://www.boxofficemojo.com" 
url_details = f"/year/world/{anno}/" 
full_url = base_url + url_details 

session = requests.Session()
session.headers.update(headers)
request = session.get(full_url, headers=headers)
soup = BeautifulSoup(request.content, "html.parser")

#trova tutte le righe
widemovie_containers = soup.find_all("tr")

#per ogni riga
for wdc in widemovie_containers:

    #trova il titolo
    wide_title = wdc.find("td", class_ = "a-text-left mojo-field-type-release_group")

    #trova la classe del link
    if wide_title:
        widelink_title = wide_title.find("a", class_= "a-link-normal")

        #trova il link 
        if widelink_title:
            widemovie_link = widelink_title.get("href")
            widelast_link = widemovie_link.split("?")[0]

            #aggiunge il link nella lista
            if widelast_link:
                worldlinks.append({ "Link Worldwide" : widelast_link
                                        })
                
            #al contrario del caso italiano, nella classifica internazionale non ci sono duplicati
            #quindi posso prendere semplicemente tutti i link senza distinzioni

                   
#SECONDO SCRAPING

#per ogni link trovato in worldlinks, trovo il link della sezione che mi serve
for worldlink in worldlinks:

    base_url = "https://www.boxofficemojo.com"
    full_url4 = base_url + worldlink["Link Worldwide"]

    request4 = session.get(full_url4, headers=headers, timeout=30)
    soup4 = BeautifulSoup(request4.content, "html.parser")

    #trovo il div esatto del link
    moviesection_link = soup4.find("div", class_= "a-section a-spacing-none mojo-title-release-refiner-option")
    
    #trovo la classe del link
    if moviesection_link:
        movieinfo_linktag= moviesection_link.find("a", class_= "a-link-normal mojo-title-link refiner-display-highlight")

        #trovo il link definitivo 
        if movieinfo_linktag:
            movieinfo_link = movieinfo_linktag.get("href")
            movielast_link = movieinfo_link.split("?")[0]

            #se il link esiste, lo aggiungo alla lista
            if movielast_link:
                lastworldlinks.append ({ "Definitive Link Worldwide" : movielast_link
                                    })
                time.sleep(1)

#TERZO E ULTIMO SCRAPING
#dalla lista di link lastworldlinks, mi prendo le informazioni che mi servono

for lastlink in lastworldlinks:
  
    base_url = "https://www.boxofficemojo.com"
    full_url5 = base_url + lastlink["Definitive Link Worldwide"]

    request5 = session.get(full_url5, headers=headers, timeout=30)
    soup5 = BeautifulSoup(request5.content, "html.parser")

    #titolo film
    widemovie_titletag = soup5.find("h1", class_= "a-size-extra-large")
    
    if widemovie_titletag:
        widemovie_title = widemovie_titletag.contents[0].strip()

    #sezione div 
        wide_containers = soup5.find("div", class_="a-section a-spacing-none mojo-gutter mojo-summary-table")

        if wide_containers:
            wide_table = wide_containers.find("div", class_= "a-section a-spacing-none mojo-summary-values mojo-hidden-from-mobile")
       
            if wide_table:
                widetag_distributor = wide_table.find("span", string= "Domestic Distributor")

                if widetag_distributor:
                    widedistributor_container = widetag_distributor.find_next("span")
                    first_text = widedistributor_container.find(string=True)

                    if first_text:
                        widemovie_distributor = first_text.strip()
                    else: 
                        widemovie_distributor = "Not classified"

                else:
                    widemovie_distributor = "Not classified"
                
                widetag_genre = wide_table.find("span", string="Genres")
                if widetag_genre:
                    widegeneral_genre = widetag_genre.find_next("span").get_text()
                    wideclean_genre = widegeneral_genre.split()
                    widemovie_genre = ", ".join(wideclean_genre)
                else:
                    widemovie_genre = "Not classified"

                worldmovie_dati.append({ "title" : widemovie_title.upper(),
                                         "distributor" : widemovie_distributor,
                                         "genre" : widemovie_genre
                                        })
                time.sleep(1)

widemovies = pd.DataFrame(worldmovie_dati)

widemovies.to_csv("dati_grezzi_scraping/anagrafica_int2025.csv", encoding="utf-8", index=False)

widemovies 


,title,distributor,genre
0,NE ZHA 2,CMC Pictures,"Action, Adventure, Animation, Comedy, Drama, F..."
1,ZOOTOPIA 2,Walt Disney Studios Motion Pictures,"Action, Adventure, Animation, Comedy, Crime, F..."
2,AVATAR: FIRE AND ASH,20th Century Studios,"Action, Adventure, Animation, Fantasy, Sci-Fi"
3,LILO & STITCH,Walt Disney Studios Motion Pictures,"Action, Adventure, Comedy, Drama, Family, Fant..."
4,A MINECRAFT MOVIE,Warner Bros.,"Action, Adventure, Comedy, Family, Fantasy"
...,...,...,...
192,GRACE,MUBI,Drama
193,SISU: ROAD TO REVENGE,Sony Pictures Releasing,"Action, War"
194,CHASSE GARDÉE 2,Not classified,Comedy
195,DETECTIVE KIEN: THE HEADLESS HORROR,Not classified,"Drama, Horror, Mystery, Thriller"


In [15]:
#CONCATENAZIONE ANAGRAFICA INTERNAZIONALE 
#dato che ogni dataframe è diviso per anno, e si vuole un'anagrafica completa, si devono concatenare tutti i dataframe anno per anno

url2015 = "dati_grezzi_scraping/anagrafica_int2015.csv"
url2016 = "dati_grezzi_scraping/anagrafica_int2016.csv"
url2017 = "dati_grezzi_scraping/anagrafica_int2017.csv"
url2018 = "dati_grezzi_scraping/anagrafica_int2018.csv"
url2019 = "dati_grezzi_scraping/anagrafica_int2019.csv"
url2020 = "dati_grezzi_scraping/anagrafica_int2020.csv"
url2021 = "dati_grezzi_scraping/anagrafica_int2021.csv"
url2022 = "dati_grezzi_scraping/anagrafica_int2022.csv"
url2023 = "dati_grezzi_scraping/anagrafica_int2023.csv"
url2024 = "dati_grezzi_scraping/anagrafica_int2024.csv"
url2025 = "dati_grezzi_scraping/anagrafica_int2025.csv"

anagrafica2015 = pd.read_csv(url2015)
anagrafica2016 = pd.read_csv(url2016)
anagrafica2017 = pd.read_csv(url2017)
anagrafica2018 = pd.read_csv(url2018)
anagrafica2019 = pd.read_csv(url2019)
anagrafica2020 = pd.read_csv(url2020)
anagrafica2021 = pd.read_csv(url2021)
anagrafica2022 = pd.read_csv(url2022)
anagrafica2023 = pd.read_csv(url2023)
anagrafica2024 = pd.read_csv(url2024)
anagrafica2025 = pd.read_csv(url2025)

wide_full_anagrafica = pd.concat([anagrafica2015, anagrafica2016, anagrafica2017, anagrafica2018, anagrafica2019, anagrafica2020, anagrafica2021, anagrafica2022, anagrafica2023, anagrafica2024, anagrafica2025], ignore_index=True)

wide_full_anagrafica.to_csv("dati_grezzi_scraping/anagrafica_wide.csv", encoding="utf-8", index=False)

wide_full_anagrafica

,title,distributor,genre
0,STAR WARS: EPISODE VII - THE FORCE AWAKENS,Walt Disney Studios Motion Pictures,"Action, Adventure, Sci-Fi"
1,JURASSIC WORLD,Universal Pictures,"Action, Adventure, Sci-Fi"
2,FURIOUS 7,Universal Pictures,"Action, Thriller"
3,AVENGERS: AGE OF ULTRON,Walt Disney Studios Motion Pictures,"Action, Adventure, Sci-Fi"
4,MINIONS,Universal Pictures,"Adventure, Animation, Comedy, Crime, Family, S..."
...,...,...,...
2189,GRACE,MUBI,Drama
2190,SISU: ROAD TO REVENGE,Sony Pictures Releasing,"Action, War"
2191,CHASSE GARDÉE 2,Not classified,Comedy
2192,DETECTIVE KIEN: THE HEADLESS HORROR,Not classified,"Drama, Horror, Mystery, Thriller"


In [16]:
#CONCATENAZIONE ANAGRAFICA ITALIANA / INTERNAZIONALE

import requests
import numpy as np
import pandas as pd
import re
import time 
from bs4 import BeautifulSoup

wide_url = "dati_grezzi_scraping/anagrafica_wide.csv"
ita_url = "dati_grezzi_scraping/anagrafica_ita.csv"

wide_anagrafica = pd.read_csv(wide_url)
ita_anagrafica = pd.read_csv(ita_url)

full_anagrafiche = pd.concat([ita_anagrafica, wide_anagrafica], ignore_index=True)

#dato che potrebbero esserci casi in cui nella ex tabella ita il distributor non è specificato, ma nella tabella wide si
#una volta unificate le due tabelle, si deve dare priorità ai film che hanno un distributor specificato
#nel momento di rimuovere i duplicati, è bene verificare che prima siano presi i duplicati il quale distributor è specificato
#creo una tabella true/false in base al distributor uguale a "not classified", così da ordinare per prima i false e poi i true
#in questo modo quando rimuovo i duplicati, prima 

#filtro true/false in corrispondenza a "not classified"
filter = full_anagrafiche["distributor"] == "Not classified"

#aggiungo la nuova colonna filtro
full_anagrafiche["filter"] = filter

#ordino alfabeticamente
full_anagrafiche = full_anagrafiche.sort_values("filter", ascending=True)

#rimuovo i duplicati, tenendo il primo record
full_anagrafiche = full_anagrafiche.drop_duplicates(subset="title", keep="first")

#rimuovo colonna filtro
full_anagrafiche = full_anagrafiche.drop(columns=["filter"])

#ordino alfabeticamente, resettando gli indici così da assegnare nuove chiavi
full_anagrafiche = full_anagrafiche.sort_values("title", ascending=True).reset_index(drop=True)

#assegno una nuova colonna per generare un id progressivo per i film 
full_anagrafiche["ID_movie"] = [f"MOV-{i+1:04d}" for i in range(len(full_anagrafiche))]

full_anagrafiche.to_csv("dim_movies.csv", encoding="utf-8", index=False)

full_anagrafiche


,title,distributor,genre,ID_movie
0,#ALIVE,Not classified,"Action, Drama, Horror, Thriller",MOV-0001
1,10 CLOVERFIELD LANE,Paramount Pictures,"Drama, Horror, Sci-Fi, Thriller",MOV-0002
2,10 DAYS WITH DAD,Distrib Films,"Comedy, Family",MOV-0003
3,10 LIVES,Not classified,"Animation, Comedy, Family, Fantasy",MOV-0004
4,100% WOLF,Viva Pictures,"Adventure, Animation, Comedy, Family, Fantasy",MOV-0005
...,...,...,...,...
2168,ZOOLANDER 2,Paramount Pictures,"Action, Adventure, Comedy, Mystery, Romance",MOV-2169
2169,ZOOTOPIA,Walt Disney Studios Motion Pictures,"Action, Adventure, Animation, Comedy, Crime, F...",MOV-2170
2170,ZOOTOPIA 2,Walt Disney Studios Motion Pictures,"Action, Adventure, Animation, Comedy, Crime, F...",MOV-2171
2171,¡A TODO TREN! DESTINO ASTURIAS,Not classified,Comedy,MOV-2172


In [17]:
#CREAZIONE TABELLA FILM-GENERI
#la maggiorparte dei film ha più generi e non solo uno; per poter fare un'analisi successiva che riguardi il genere
#bisogna creare una nuova tabella e splittare i generi

#genero una seconda tabella basandomi sul dataset pulito
generi_anagrafica = full_anagrafiche[["title", "genre"]].copy()

#divido i generi separati da virgole
generi_anagrafica["genre"] = generi_anagrafica["genre"].str.split(", ")

#utilizzo il metodo .explode(), prende ogni elemmento e crea una nuova riga con lo stesso titolo
generi_anagrafica = generi_anagrafica.explode("genre")

generi_anagrafica

,title,genre
0,#ALIVE,Action
0,#ALIVE,Drama
0,#ALIVE,Horror
0,#ALIVE,Thriller
1,10 CLOVERFIELD LANE,Drama
...,...,...
2170,ZOOTOPIA 2,Family
2170,ZOOTOPIA 2,Mystery
2171,¡A TODO TREN! DESTINO ASTURIAS,Comedy
2172,ÉPOUSE-MOI MON POTE,Comedy


In [18]:
#CREAZIONE TABELLA GENERI UNICI
#per ogni genere preferisco assegnare una chiave univoca

#prendo i valori unici dei generi
lista_generi = generi_anagrafica["genre"].unique()

#trasformo in un dataframe
tabella_generi = pd.DataFrame(lista_generi, columns=["genre"])
#ordino alfabeticamente e resetto gli indici nuovi
tabella_generi = tabella_generi.sort_values("genre", ascending=True).reset_index(drop=True)

#creo id univoco come per i film
tabella_generi["ID_genre"] = [f"GEN-{i+1:03d}" for i in range(len(tabella_generi))]

tabella_generi.to_csv("dim_genres.csv", encoding="utf-8", index=False)

tabella_generi

,genre,ID_genre
0,Action,GEN-001
1,Adventure,GEN-002
2,Animation,GEN-003
3,Biography,GEN-004
4,Comedy,GEN-005
5,Crime,GEN-006
6,Documentary,GEN-007
7,Drama,GEN-008
8,Family,GEN-009
9,Fantasy,GEN-010


In [19]:
#ASSEGNO GLI ID UNIVOCI NELLA TABELLA GENERI_ANAGRAFICHE
#con una merge, assegno ogni id al suo genere e film

generi_anagrafica = generi_anagrafica.merge(tabella_generi, on="genre", how="left")
generi_anagrafica = generi_anagrafica.merge(full_anagrafiche[["title", "ID_movie"]], on="title", how="left")

generi_anagrafica.to_csv("bridge_moviegenre.csv", encoding="utf-8", index=False)

generi_anagrafica

,title,genre,ID_genre,ID_movie
0,#ALIVE,Action,GEN-001,MOV-0001
1,#ALIVE,Drama,GEN-008,MOV-0001
2,#ALIVE,Horror,GEN-012,MOV-0001
3,#ALIVE,Thriller,GEN-022,MOV-0001
4,10 CLOVERFIELD LANE,Drama,GEN-008,MOV-0002
...,...,...,...,...
7278,ZOOTOPIA 2,Family,GEN-009,MOV-2171
7279,ZOOTOPIA 2,Mystery,GEN-015,MOV-2171
7280,¡A TODO TREN! DESTINO ASTURIAS,Comedy,GEN-005,MOV-2172
7281,ÉPOUSE-MOI MON POTE,Comedy,GEN-005,MOV-2173


In [ ]:
#CONNESSIONE AL DATABASE
#una volta creato il database su MYSQL, connetto i csv 

import os
import dotenv
import pandas as pd
import sqlalchemy 

#carico il file env con le credenziali
dotenv.load_dotenv(dotenv_path="boxoffice_cred.env", override=True)

username = os.getenv("username")
password = os.getenv("password")
host = os.getenv("host")
dbname = os.getenv("dbname")

#creo connessione
connection_string = f"mysql+pymysql://{username}:{password}@{host}/{dbname}"

db_engine = sqlalchemy.create_engine(connection_string)

#da csv a pd
fact_int = pd.read_csv("fact_intgross.csv")
fact_ita = pd.read_csv("fact_itagross.csv")
dim_movies = pd.read_csv("dim_movies.csv")
dim_genres = pd.read_csv("dim_genres.csv")
bridge = pd.read_csv("bridge_moviegenre.csv")

#carico su sql
fact_int.to_sql("fact_intgross", con=db_engine, if_exists='append', index=False)
fact_ita.to_sql("fact_itagross", con=db_engine, if_exists='append', index=False)
dim_movies.to_sql("dim_movies", con=db_engine, if_exists='append', index=False)
dim_genres.to_sql("dim_genres", con=db_engine, if_exists='append', index=False)
bridge.to_sql("bridge_moviegenre", con=db_engine, if_exists='append', index=False)